# DART 기반 매출예측 → RIM Valuation — `dart_rim_valuation_v1`

**파이프라인**

```
korea_fs_data_from_DART_V2 (dart_korea_fs_loader_v6 적재본)
     │  PART 1  DART 계정 → 표준키 매핑 · 분기화(누적/분기 자동 판별)
     ▼
분기 매출 시계열 ──► PART 2  SARIMA · ETS · Theta · Ensemble (Korea_revenue_forecast_v4 준용)
     │                       └─► korea_revenue_forecast_from_DART  (forecast_date 별 vintage 저장)
     ▼
PART 3  DartRIMModel  (korea_rim_valuation_v5 = DuPont ROE → Ohlson RIM, Phase 2 선형 fade, TV=0 그대로)
     │            부족 항목은 DART 데이터로 추정 (주식수→시총/주가, 배당→CF 배당금지급 or Clean-surplus 역산)
     ▼
PART 4  Excel 출력  →  …\FCFF_RIM_재무데이터\{ticker}_{기업명}_RIM_DART_{YYYYMMDD}.xlsx   (DB 저장 없음)
```

| 준용 원본 | 이 노트북에서의 역할 |
|---|---|
| `dart_korea_fs_loader_v6` | 테이블 스키마(`korea_fs_data_from_DART_V2`)·quarter 라벨(Q1/H1/Q3/FY)·단위(원) |
| `Korea_revenue_forecast_v4` | `_prepare_series` / `_run_models` / long 변환 / 배치 저장 — 로직 동일, 원천·저장 테이블만 변경 |
| `korea_rim_valuation_v5` | `KoreaRIMModel` + v5 선형 fade 패치 전체 로직 — 데이터 로더만 DART 로 교체 |

> 매출 예측 테이블은 `dart_fcff_valuation_v1` 과 **같은** `korea_revenue_forecast_from_DART` 를 공유합니다 (같은 계측일 vintage 재사용).

**DART 원천의 특성 (DG 와 다른 점)**
- 금액 단위 **원** (DG 는 천원) → `FS_UNIT_MULTIPLIER = 1`
- 손익계산서(IS/CIS) `thstrm_amount` 는 보고서별 3개월치, 현금흐름표(CF) 는 누적치가 일반적 → `_detect_flow_mode()` 로 종목별 자동 판별 후 순수 분기값 산출
- 주식수 항목이 재무제표 API 에 없음 → `시가총액(ks_listed_company_daily_marketcap) ÷ 현재가` 로 추정
- 계정 ID 가 표준(ifrs-full_*)·비표준("-표준계정코드 미사용-") 혼재 → `DART_ACCOUNT_MAP` 이 **account_id 우선, account_nm 정규식 보조** 로 매핑

실행 순서: **Cell 1 → Cell 2(입력) → 위에서 아래로 순서대로**

## PART 1 · 환경
### Cell 1 · 경로 자동 감지 (노트북 / 데스크탑 공용)

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    """DATA 폴더의 부모를 프로젝트 루트로 sys.path 에 등록 (루트 + DATA 둘 다)."""
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            for q in (root, str(p / "DATA")):
                if q not in sys.path:
                    sys.path.insert(0, q)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(os.path.join(cand, "DATA")):
            for q in (cand, os.path.join(cand, "DATA")):
                if q not in sys.path:
                    sys.path.insert(0, q)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 수정하세요.")

_ROOT = _setup_path()
print(f"[확인] DATA 경로 : {os.path.join(_ROOT, 'DATA')}")

[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


### Cell 2 · ★ 입력 변수 (여기만 수정)

In [2]:
# ═══════════════════════════════════════════════════════════════
#  ★★★ Control Panel — 이 노트북의 모든 사용자 입력은 여기 한 곳 ★★★
# ═══════════════════════════════════════════════════════════════
from datetime import date

# ── ① 평가 대상 ────────────────────────────────────────────────
TICKERS = [                 # DART 형식 6자리 ('A' 접두사 있어도 자동 제거)
    "278470",               #   SK하이닉스
    # "278470",             #   에이피알
]
VERBOSE = True

# ── ② 출력 폴더 (요구사항 7) — 첫 번째로 존재하는 부모 폴더 채택 ──
_EXPORT_DIR_CANDIDATES = [
    r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터",          # 랩탑
    r"C:\Users\Hoyoung_Park\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터",   # 데스크탑
]
EXPORT_FILE_TAG = "RIM_DART"        # 파일명에 RIM + DART 표시
CORP_NAMES = {                      # 파일명용 기업명 (없으면 시총 테이블에서 시도, 실패 시 'NA')
    "000660": "SK하이닉스", "005930": "삼성전자", "278470": "에이피알",
}

# ── ③ DB 테이블 ────────────────────────────────────────────────
TABLE_DART_FS      = "korea_fs_data_from_DART_V3"        # 원천 (loader_v6)
TABLE_DART_FC      = "korea_revenue_forecast_from_DART"  # ★ 신규: DART 매출예측 (vintage)
TABLE_MARKETCAP    = "ks_listed_company_daily_marketcap"
TABLE_PRICE        = "KSE_Price"

# ── ④ 매출 예측 (Korea_revenue_forecast_v4 준용) ─────────────
FORECAST_QUARTERS  = 8          # 예측 분기 수
MIN_DATA_PERIODS   = 20         # 최소 연속 분기 (DART 는 2015~ 이므로 24 → 20 완화)
ENSEMBLE_AGG       = "median"   # 'median' | 'mean'
FORECAST_DATE      = date.today()   # ★ 계측일 (vintage key). 과거 날짜 지정 시 그 날짜로 저장
RERUN_FORECAST     = True       # False → 같은 forecast_date 에 이미 저장된 예측이 있으면 재예측 생략
FORECAST_MODULE_CANDIDATES = ["universal_ts_forecast_function_v3",
                              "universal_ts_forecast_function_v2",
                              "universal_ts_forecast_function"]

# ── ⑤ DART 분기화 방식 ────────────────────────────────────────
#   'auto'       : 연도별 (Q1+H1+Q3)/FY 비율로 누적/분기 자동 판별 (권장)
#   'quarter'    : thstrm_amount 가 각 보고서의 3개월치 (Q4 = FY − Q1−Q2−Q3)
#   'cumulative' : thstrm_amount 가 누적치 (Q2 = H1 − Q1 …)
IS_FLOW_MODE = "auto"
CF_FLOW_MODE = "auto"

# ── ⑥ 모델 파라미터 (korea_rim_valuation_v5 와 동일) ─────────
FORECAST_HORIZON     = FORECAST_QUARTERS
MIN_HISTORY          = 12           # DuPont OLS 최소 분기
MIN_REVENUE_QUARTERS = MIN_DATA_PERIODS
OLS_MIN_R2           = 0.3
OLS_MIN_SAMPLES      = 12
WINSORIZE_LIMITS     = (0.05, 0.95)
GDP_GROWTH           = 0.04
TV_RE_GAP            = 0.010        # (v5 선형 fade 에서는 TV 가 구조적으로 0 — 참고용)
RETENTION_FLOOR      = -0.50
BV_RETENTION_CAP     = 0.75
PHASE1_BASE_YR       = 2
PHASE1_EXTRA_YR      = {"Exceptional moat": 5, "Wide moat": 4, "Wide-Narrow moat": 3,
                        "Narrow moat": 2, "Some moat": 1, "No moat": 0, "Unknown (fallback)": 1}
PHASE1_YR_BY_MOAT    = {g: PHASE1_BASE_YR + e for g, e in PHASE1_EXTRA_YR.items()}
PHASE1_ROE_FLOOR_RATIO   = 0.80
PHASE1_ROE_CEILING_RATIO = 2.50

# ── ⑦ Re / ERP ─────────────────────────────────────────────────
ERP_METHOD       = "damodaran_floor"
DAMODARAN_ERP_KR = 0.07
GEO_FLOOR        = 0.07
RF_FALLBACK      = 0.035
RD_DEFAULT       = 0.045
RD_ANNUALIZE_QUARTERLY = True   # EVA 용 간이 WACC 의 Rd: 분기 이자비용 ×4 연율화
RE_FLOOR, RE_CAP = None, None    # v2 옵션 A: Re clip 비활성 (FCFF 와 일관)

# ── ⑨ Net Debt / NWC 구성 (표준키) ───────────────────────────
DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds", "long_term_debt", "lease_liab"]
CASH_KEYS = ["cash", "short_term_invest"]

# ── ⑩ 단위 ────────────────────────────────────────────────────
MARKETCAP_UNIT_MULTIPLIER = 1_000_000   # 시총 테이블: 백만원 → 원
FS_UNIT_MULTIPLIER        = 1           # ★ DART 는 원 단위 (DG 천원과 다름)

# ── 출력 폴더 확정 ────────────────────────────────────────────
def _pick_export_dir() -> Path:
    for c in _EXPORT_DIR_CANDIDATES:
        p = Path(c)
        if p.exists() or p.parent.exists():
            p.mkdir(parents=True, exist_ok=True)
            return p
    p = Path.home() / "FCFF_RIM_재무데이터"
    p.mkdir(parents=True, exist_ok=True)
    print(f"[WARN] 후보 경로 없음 → {p} 사용")
    return p

EXPORT_DIR = _pick_export_dir()
TICKERS = [str(t).strip().upper().lstrip("A").zfill(6) for t in TICKERS]

print("[OK] Control Panel")
print(f"  대상 종목      : {TICKERS}")
print(f"  Excel 폴더     : {EXPORT_DIR}")
print(f"  forecast_date  : {FORECAST_DATE}   예측 {FORECAST_QUARTERS}Q   최소 {MIN_DATA_PERIODS}Q   앙상블 {ENSEMBLE_AGG}")
print(f"  원천 → 예측 DB : {TABLE_DART_FS} → {TABLE_DART_FC}")

[OK] Control Panel
  대상 종목      : ['278470']
  Excel 폴더     : C:\Users\82108\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터
  forecast_date  : 2026-08-25   예측 8Q   최소 20Q   앙상블 median
  원천 → 예측 DB : korea_fs_data_from_DART_V2 → korea_revenue_forecast_from_DART


### Cell 3 · Import & DB 연결

In [3]:
import gc, re, math, time, traceback, importlib
from datetime import datetime, timedelta
from typing import Optional, Dict, Any, List, Tuple, Union

import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
from tqdm.auto import tqdm
from sqlalchemy import text as _sa_text

# ── 내부 모듈 ───────────────────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    to_dg_ticker, to_price_ticker, get_pymysql_conn,
    load_korea_marketcap_latest, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, DataQualityReport,
)

# 예측 모듈: 후보 순서대로 import (v4 노트북과 동일한 폴백 방식)
_fc_mod = None
for _name in FORECAST_MODULE_CANDIDATES:
    try:
        _fc_mod = importlib.import_module(_name)
        FORECAST_MODULE = _name
        break
    except ModuleNotFoundError:
        continue
if _fc_mod is None:
    raise ImportError(f"예측 모듈을 찾을 수 없습니다: {FORECAST_MODULE_CANDIDATES}")
forecast_sarima = _fc_mod.forecast_sarima
forecast_ets    = _fc_mod.forecast_ets
forecast_theta  = _fc_mod.forecast_theta
infer_freq_alias          = getattr(_fc_mod, "infer_freq_alias", None)
seasonal_periods_from_freq = getattr(_fc_mod, "seasonal_periods_from_freq", None)
clear_memory              = getattr(_fc_mod, "clear_memory", lambda: gc.collect())

def log(tag: str, msg: str):
    print(f"[{datetime.now():%H:%M:%S}][{tag}] {msg}", flush=True)

db_info = get_db_info()
engine  = get_engine(db_info)
with engine.connect() as c:
    c.execute(_sa_text("SELECT 1"))
log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
log("MOD", f"예측 모듈 = {FORECAST_MODULE}")

[16:02:51][DB] 연결 성공 host=192.168.0.230 port=3307
[16:02:51][MOD] 예측 모듈 = universal_ts_forecast_function_v3


### Cell 4 · 시장 파라미터 (Rf · KOSPI · E(Rm)) — v5 동일

In [4]:
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
KOSPI_PX = load_kospi_series(start_date=(datetime.today() - timedelta(days=365*12)).strftime("%Y-%m-%d"))
mkt = estimate_market_return(method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX, years=10,
                             damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR)
E_RM, ERP = mkt["e_rm"], mkt["erp"]
log("MKT", f"Rf={RF:.3%} ({RF_SOURCE})  ERP={ERP:.3%}  E(Rm)={E_RM:.3%}  KOSPI={KOSPI_PX.iloc[-1]:,.1f}")

[KOSPI] try source=fdr (2014-08-28 ~ 2026-08-25) ... FAIL (FDR 실패 (재시도 3회): LOGOUT)
[KOSPI] try source=pykrx (2014-08-28 ~ 2026-08-25) ... FAIL ('지수명')
[KOSPI] try source=yfinance (2014-08-28 ~ 2026-08-25) ... OK  2,936거래일
[16:03:02][MKT] Rf=4.335% (BOK_2026-08-24)  ERP=7.000%  E(Rm)=11.335%  KOSPI=6,697.0


### Cell 5 · DART 계정 → 표준키 매핑 & 분기화 로더

`DART_ACCOUNT_MAP[std_key]` = 컴포넌트 리스트. 각 컴포넌트는 **account_id 후보(우선) → account_nm 정규식(보조)** 순으로
첫 매칭 계정 하나만 채택(중복합산 방지)하고, 컴포넌트끼리는 합산(예: 리스부채 유동+비유동).

`flow` : `stock`(BS 시점값) / `is`(손익 흐름) / `cf`(현금흐름 흐름). 흐름 항목은 `_detect_flow_mode()` 결과에 따라 순수 분기값으로 변환.

In [5]:
# ── 표준키 정의 ────────────────────────────────────────────────
#   comp : 컴포넌트 리스트(합산). 각 컴포넌트의 ids 후보는 기간별로 coalesce (택소노미 변경 대응:
#          ifrs-full_X 를 쓰면 ifrs_X 도 자동 후보). nm 은 ids 모두 없을 때만 정규식 보조.
#   alts : 대안 그룹 리스트 — 앞 그룹이 하나도 안 잡힐 때만 다음 그룹 사용 (중복합산 방지)
DART_ACCOUNT_MAP: Dict[str, Dict[str, Any]] = {
    # ── 손익 (IS → CIS) ──────────────────────────────────────
    "revenue":          {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["ifrs-full_Revenue"], "nm": r"^(매출액|매출|수익\(매출액\)|영업수익)$"}]},
    "cost_of_sales":    {"flow": "is", "sj": ["IS", "CIS"], "comp": [{"ids": ["ifrs-full_CostOfSales"], "nm": r"^매출원가$"}]},
    "operating_income": {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["dart_OperatingIncomeLoss", "ifrs-full_ProfitLossFromOperatingActivities"], "nm": r"^영업이익(\(손실\))?$"}]},
    "pretax_income":    {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["ifrs-full_ProfitLossBeforeTax"], "nm": r"^법인세(비용)?차감전(순)?(이익|손익)"}]},
    "tax_expense":      {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["ifrs-full_IncomeTaxExpenseContinuingOperations"], "nm": r"^법인세(비용|비용\(수익\)|비용\(이익\)|수익\(비용\))$"}]},
    "net_income":       {"flow": "is", "sj": ["IS", "CIS"], "alts": [
        [{"ids": ["ifrs-full_ProfitLoss"], "nm": r"^(연결)?(당기|분기|반기|당분기|당반기)?순(이익|손익|손실)(\(손실\)|\(이익\))?$"}],
        [{"ids": ["ifrs-full_ProfitLossAttributableToOwnersOfParent"]},          # 지배 + 비지배 합산
         {"ids": ["ifrs-full_ProfitLossAttributableToNoncontrollingInterests"]}],
    ]},
    "net_income_parent": {"flow": "is", "sj": ["IS", "CIS"], "comp": [{"ids": ["ifrs-full_ProfitLossAttributableToOwnersOfParent"]}]},
    "interest_expense": {"flow": "is", "sj": ["IS", "CIS"], "comp": [
        {"ids": ["dart_InterestExpenseFinanceCosts", "ifrs-full_InterestExpense"], "nm": r"^이자비용$"}]},
    "finance_costs":    {"flow": "is", "sj": ["IS", "CIS"], "comp": [{"ids": ["ifrs-full_FinanceCosts"], "nm": r"^(금융비용|금융원가)$"}]},
    # ── 현금흐름 ─────────────────────────────────────────────
    "da_cf":            {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_AdjustmentsForDepreciationExpense", "dart_DepreciationExpenseCashFlow"], "nm": r"^감가상각비$"}]},
    "intangible_amort_cf": {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_AdjustmentsForAmortisationExpense", "dart_AmortisationExpenseCashFlow"], "nm": r"^무형자산상각비?$"}]},
    "capex_tangible":   {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities", "dart_PurchaseOfPropertyPlantAndEquipment"],
         "nm": r"^유형자산의?(취득|증가)$"}]},
    "capex_intangible": {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities", "dart_PurchaseOfIntangibleAssets"],
         "nm": r"^무형자산의?(취득|증가)$"}]},
    "interest_paid_cf": {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_InterestPaidClassifiedAsOperatingActivities", "ifrs-full_InterestPaidClassifiedAsFinancingActivities"],
         "nm": r"^이자(의)?지급$"}]},
    "cfo":              {"flow": "cf", "sj": ["CF"], "comp": [{"ids": ["ifrs-full_CashFlowsFromUsedInOperatingActivities"], "nm": r"^영업활동현금흐름$"}]},
    "dividends_paid":   {"flow": "cf", "sj": ["CF"], "comp": [
        {"ids": ["ifrs-full_DividendsPaidClassifiedAsFinancingActivities", "dart_DividendsPaid"], "nm": r"^(현금)?배당금(의)?지급$"}]},
    # ── 재무상태표 ───────────────────────────────────────────
    "cash":             {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_CashAndCashEquivalents"], "nm": r"^현금및현금성자산$"}]},
    "short_term_invest": {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermDepositsNotClassifiedAsCashEquivalents", "ifrs-full_ShorttermDepositsNotClassifiedAsCashEquivalents"], "nm": r"^단기금융상품$"},
        {"ids": ["ifrs-full_CurrentInvestments", "dart_CurrentAvailableForSaleFinancialAssets", "dart_ShortTermTradingFinancialAssets"], "nm": r"^단기투자자산$"}]},
    "receivables":      {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermTradeReceivable", "ifrs-full_CurrentTradeReceivables", "ifrs-full_TradeAndOtherCurrentReceivables"],
         "nm": r"^(매출채권|매출채권및기타채권|매출채권및기타유동채권)$"}]},
    "inventories":      {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_Inventories"], "nm": r"^재고자산$"}]},
    "prepaid_expenses": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["dart_ShortTermPrepaidExpenses"], "nm": r"^선급비용$"}]},
    "payables":         {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermTradePayables", "ifrs-full_TradeAndOtherCurrentPayablesToTradeSuppliers", "ifrs-full_TradeAndOtherCurrentPayables"],
         "nm": r"^(매입채무|매입채무및기타채무|매입채무및기타유동채무)$"}]},
    "accrued_expenses": {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["dart_ShortTermAccruedExpenses", "dart_CurrentNontradePayables"], "nm": r"^(미지급비용|기타지급채무)$"}]},
    "other_payables":   {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["dart_ShortTermOtherPayables"], "nm": r"^(미지급금|단기미지급금)$"}]},
    "advances_received": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["dart_ShortTermAdvancesCustomers"], "nm": r"^선수금$"}]},
    "contract_liabilities": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_CurrentContractLiabilities"], "nm": r"^(계약부채|유동계약부채)$"}]},
    "short_term_debt":  {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_ShorttermBorrowings", "dart_ShortTermBorrowings", "ifrs-full_CurrentBorrowingsAndCurrentPortionOfNoncurrentBorrowings"],
         "nm": r"^(단기차입금|단기차입금및유동성장기차입금)$"}]},
    "current_lt_debt":  {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_CurrentPortionOfLongtermBorrowings", "dart_CurrentPortionOfLongTermBorrowingsAndDebentures"],
         "nm": r"^유동성(장기차입금|장기부채|사채|장기차입부채)"}]},
    "bonds":            {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_BondsIssued", "dart_BondsIssued", "ifrs-full_DebenturesIssued"], "nm": r"^(사채|비유동사채|회사채)$"}]},
    "long_term_debt":   {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_LongtermBorrowings", "dart_LongTermBorrowingsGross"], "nm": r"^(장기차입금|비유동차입금)$"}]},
    "lease_liab":       {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_CurrentLeaseLiabilities", "dart_CurrentLeaseLiabilities"], "nm": r"^(유동리스부채|리스부채\(유동\))$"},
        {"ids": ["ifrs-full_NoncurrentLeaseLiabilities", "dart_NoncurrentLeaseLiabilities"], "nm": r"^(비유동리스부채|리스부채\(비유동\)|장기리스부채)$"}]},
    "total_equity":     {"flow": "stock", "sj": ["BS"], "alts": [
        [{"ids": ["ifrs-full_Equity"], "nm": r"^자본총계$"}],
        [{"ids": ["ifrs-full_EquityAttributableToOwnersOfParent"]}, {"ids": ["ifrs-full_NoncontrollingInterests"]}],
    ]},
    "equity_parent":    {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_EquityAttributableToOwnersOfParent"]}]},
    "total_assets":     {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_Assets"], "nm": r"^자산총계$"}]},
    "total_liabilities": {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_Liabilities"], "nm": r"^부채총계$"}]},
    "ppe":              {"flow": "stock", "sj": ["BS"], "comp": [{"ids": ["ifrs-full_PropertyPlantAndEquipment"], "nm": r"^유형자산$"}]},
    "intangibles":      {"flow": "stock", "sj": ["BS"], "comp": [
        {"ids": ["ifrs-full_IntangibleAssetsOtherThanGoodwill", "ifrs-full_IntangibleAssetsAndGoodwill", "dart_GoodwillGross"], "nm": r"^무형자산$"}]},
}

_Q_ORDER = {"Q1": 1, "H1": 2, "Q3": 3, "FY": 4}


def _expand_ids(ids: List[str]) -> List[str]:
    """ifrs-full_X → [ifrs-full_X, ifrs_X] 자동 확장 (구 택소노미 대응)."""
    out = []
    for i in ids:
        out.append(i)
        if i.startswith("ifrs-full_"):
            out.append("ifrs_" + i[len("ifrs-full_"):])
    return list(dict.fromkeys(out))


def load_dart_long(ticker: str, db_info: dict, table: str = TABLE_DART_FS) -> pd.DataFrame:
    """DART 원천 long 테이블 조회 (ticker 6자리). SQLAlchemy engine 경유 (pymysql DictCursor 와 read_sql 충돌 회피)."""
    sql = _sa_text(f"""
        SELECT corp_code, bsns_year, reprt_code, quarter, sj_div, account_id, account_nm,
               thstrm_amount, report_date, ticker
        FROM {table}
        WHERE ticker = :t AND thstrm_amount IS NOT NULL
        ORDER BY bsns_year, reprt_code
    """)
    with engine.connect() as conn:
        df = pd.read_sql(sql, conn, params={"t": ticker})
    if df.empty:
        return df
    # 방어: 헤더가 데이터로 들어온 경우 제거
    df = df[df["report_date"].astype(str) != "report_date"].copy()
    df["account_nm_norm"] = df["account_nm"].astype(str).str.replace(r"\s+", "", regex=True)
    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
    df["bsns_year"] = pd.to_numeric(df["bsns_year"], errors="coerce").astype(int)
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    return df


def _resolve_component(df: pd.DataFrame, comp: Dict, sj_list: List[str]) -> List[Tuple[str, str]]:
    """컴포넌트 → [(token, how), ...]  (기간별 coalesce 순서). ids 는 우선순위대로 모두 채택, 없으면 nm 정규식."""
    sub = df[df["sj_div"].isin(sj_list)]
    toks = [(aid, "id") for aid in _expand_ids(comp.get("ids", [])) if (sub["account_id"] == aid).any()]
    if not toks and comp.get("nm"):
        m = sub[sub["account_nm_norm"].str.match(comp["nm"])]
        if not m.empty:
            grp = m.groupby(["account_id", "account_nm"]).size().sort_values(ascending=False)
            toks = [(f"NM::{aid}::{anm}", "nm") for (aid, anm) in grp.index]
    return toks


def _component_series(df: pd.DataFrame, toks: List[Tuple[str, str]], sj_list: List[str]) -> pd.DataFrame:
    """토큰 리스트를 우선순위대로 coalesce → [bsns_year, quarter, report_date, value]"""
    sub = df[df["sj_div"].isin(sj_list)]
    sub = sub.assign(_sj_rank=sub["sj_div"].map({s: i for i, s in enumerate(sj_list)}))
    parts = []
    for rank, (tok, _) in enumerate(toks):
        if tok.startswith("NM::"):
            _, aid, anm = tok.split("::", 2)
            x = sub[(sub["account_id"] == aid) & (sub["account_nm"] == anm)]
        else:
            x = sub[sub["account_id"] == tok]
        parts.append(x.assign(_rank=rank))
    if not parts:
        return pd.DataFrame(columns=["bsns_year", "quarter", "report_date", "value"])
    allx = pd.concat(parts).sort_values(["bsns_year", "quarter", "_rank", "_sj_rank"])
    allx = allx.drop_duplicates(["bsns_year", "quarter"])       # 기간별 첫 우선순위 채택
    return allx[["bsns_year", "quarter", "report_date", "thstrm_amount"]].rename(columns={"thstrm_amount": "value"})


def _detect_flow_mode(pivot: pd.DataFrame, verbose: bool = False, label: str = "") -> str:
    """
    연도별 (Q1+H1+Q3)/FY 비율의 중앙값으로 누적/분기 판별.
      분기값이면 ≈ 0.75, 누적값이면 ≈ 1.5 (1/4+2/4+3/4).  임계 1.10.
    pivot: index=bsns_year, columns=Q1/H1/Q3/FY
    """
    need = ["Q1", "H1", "Q3", "FY"]
    if not all(c in pivot.columns for c in need):
        return "quarter"
    full = pivot.dropna(subset=need)
    full = full[(full[need].abs() > 0).all(axis=1)]
    if full.empty:
        return "quarter"
    ratio = ((full["Q1"] + full["H1"] + full["Q3"]) / full["FY"]).abs()
    med = float(ratio.median())
    mode = "cumulative" if med > 1.10 else "quarter"
    if verbose:
        log("FLOW", f"{label}: median (Q1+H1+Q3)/FY = {med:.2f} (n={len(full)}) → {mode}")
    return mode


def _flows_to_quarterly(pivot: pd.DataFrame, mode: str) -> pd.DataFrame:
    """index=bsns_year, columns Q1/H1/Q3/FY → 순수 분기값 columns Q1..Q4"""
    out = pd.DataFrame(index=pivot.index, columns=["Q1", "Q2", "Q3", "Q4"], dtype=float)
    q1 = pivot.get("Q1"); h1 = pivot.get("H1"); q3 = pivot.get("Q3"); fy = pivot.get("FY")
    if mode == "cumulative":
        out["Q1"] = q1
        out["Q2"] = h1 - q1 if (h1 is not None and q1 is not None) else np.nan
        out["Q3"] = q3 - h1 if (q3 is not None and h1 is not None) else np.nan
        out["Q4"] = fy - q3 if (fy is not None and q3 is not None) else np.nan
    else:  # quarter
        out["Q1"] = q1
        out["Q2"] = h1
        out["Q3"] = q3
        out["Q4"] = fy - (q1 + h1 + q3) if all(x is not None for x in (fy, q1, h1, q3)) else np.nan
    return out


def _quarter_end(year: int, q: int) -> pd.Timestamp:
    return pd.Period(f"{year}Q{q}", freq="Q").to_timestamp(how="end").normalize()


def load_dart_financials_wide(ticker: str, db_info: dict, verbose: bool = False,
                              is_mode: str = IS_FLOW_MODE, cf_mode: str = CF_FLOW_MODE
                              ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    DART long → 분기 wide (index=분기말 Timestamp, columns=표준키, 단위 원).
    Returns (wide, mapping_df)  mapping_df: 표준키별 채택 계정·매칭방식·flow 모드.
    """
    ticker = str(ticker).upper().lstrip("A").zfill(6)
    df = load_dart_long(ticker, db_info)
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    def _comp_groups(spec: Dict) -> List[List[Dict]]:
        return spec["alts"] if "alts" in spec else [spec["comp"]]

    def _resolve_group(spec: Dict) -> List[Tuple[Dict, List[Tuple[str, str]]]]:
        """alts 중 하나라도 잡히는 첫 그룹의 [(comp, toks)] 반환"""
        for grp in _comp_groups(spec):
            got = [(c, _resolve_component(df, c, spec["sj"])) for c in grp]
            if any(t for _, t in got):
                return got
        return []

    # ── 1) 흐름 모드 판별 (IS: revenue, CF: da_cf → capex_tangible → cfo) ──
    def _pivot_for(std_key: str) -> Optional[pd.DataFrame]:
        spec = DART_ACCOUNT_MAP[std_key]; got = _resolve_group(spec)
        for c, toks in got:
            if toks:
                s = _component_series(df, toks, spec["sj"])
                return s.pivot(index="bsns_year", columns="quarter", values="value")
        return None

    if is_mode == "auto":
        p = _pivot_for("revenue")
        is_mode = _detect_flow_mode(p, verbose, "IS(revenue)") if p is not None else "quarter"
    if cf_mode == "auto":
        cf_mode = "cumulative"
        for k in ("da_cf", "capex_tangible", "cfo"):
            p = _pivot_for(k)
            if p is not None:
                cf_mode = _detect_flow_mode(p.abs(), verbose, f"CF({k})"); break

    # ── 2) 표준키별 시계열 구성 ────────────────────────────────
    series_map: Dict[str, pd.Series] = {}
    mapping_rows = []
    for std_key, spec in DART_ACCOUNT_MAP.items():
        total: Optional[pd.Series] = None
        mode = "stock" if spec["flow"] == "stock" else (is_mode if spec["flow"] == "is" else cf_mode)
        for comp, toks in _resolve_group(spec):
            if not toks:
                continue
            s = _component_series(df, toks, spec["sj"])
            pv = s.pivot(index="bsns_year", columns="quarter", values="value")
            for lab in ("Q1", "H1", "Q3", "FY"):
                if lab not in pv.columns:
                    pv[lab] = np.nan
            if spec["flow"] == "stock":
                qdf = pd.DataFrame({"Q1": pv["Q1"], "Q2": pv["H1"], "Q3": pv["Q3"], "Q4": pv["FY"]}, index=pv.index)
            else:
                qdf = _flows_to_quarterly(pv, mode)
            long = qdf.reset_index().melt(id_vars=qdf.index.name or "index", var_name="qn", value_name="value")
            long.columns = ["bsns_year", "qn", "value"]
            long = long.dropna(subset=["value"])
            long["date"] = [_quarter_end(int(y), int(q[1])) for y, q in zip(long["bsns_year"], long["qn"])]
            ser = long.set_index("date")["value"].astype(float)
            total = ser if total is None else total.add(ser, fill_value=0.0)
            disp = " | ".join(t if how == "id" else t.split("::", 2)[2] + f"[{t.split('::', 2)[1]}]" for t, how in toks)
            mapping_rows.append({"std_key": std_key, "component": disp, "match": toks[0][1], "flow": spec["flow"], "mode": mode,
                                 "n_q": int(ser.notna().sum())})
        if total is not None:
            series_map[std_key] = total
        else:
            mapping_rows.append({"std_key": std_key, "component": "(없음)", "match": "-", "flow": spec["flow"], "mode": "-", "n_q": 0})

    wide = pd.DataFrame(series_map).sort_index().dropna(how="all")

    # ── 3) DART 기반 보완 추정 (요구사항 5) ────────────────────
    def _est(key, note, flow="derived"):
        mapping_rows.append({"std_key": key, "component": note, "match": "est", "flow": flow, "mode": "derived",
                             "n_q": int(wide[key].notna().sum()) if key in wide.columns else 0})
    def _missing(k):
        return k not in wide.columns or wide[k].dropna().empty
    def _fill(k, ser, note):
        if k in wide.columns:
            wide[k] = wide[k].where(wide[k].notna(), ser)
        else:
            wide[k] = ser
        _est(k, note)
    #   순이익 없음 → 세전이익 − 법인세
    if _missing("net_income") and not _missing("pretax_income") and "tax_expense" in wide.columns:
        _fill("net_income", wide["pretax_income"] - wide["tax_expense"].fillna(0), "← pretax_income − tax_expense (대체 추정)")
    #   자본총계 결측 기간 → 자산총계 − 부채총계
    if "total_assets" in wide.columns and "total_liabilities" in wide.columns:
        eq_alt = wide["total_assets"] - wide["total_liabilities"]
        if _missing("total_equity") or wide["total_equity"].isna().any():
            _fill("total_equity", eq_alt, "← total_assets − total_liabilities (결측 기간 보완)")
    #   이자비용 없음 → CF 이자의 지급 → 금융비용
    if _missing("interest_expense"):
        if not _missing("interest_paid_cf"):
            _fill("interest_expense", wide["interest_paid_cf"].abs(), "← CF 이자의 지급 (대체 추정)")
        elif not _missing("finance_costs"):
            _fill("interest_expense", wide["finance_costs"], "← finance_costs (대체 추정)")
    #   CapEx 없음 → ΔPPE + D&A
    if _missing("capex_tangible") and "ppe" in wide.columns:
        da = wide["da_cf"].fillna(0) if "da_cf" in wide.columns else 0.0
        _fill("capex_tangible", (wide["ppe"].diff() + da).clip(lower=0), "← ΔPPE + D&A (대체 추정)")
    #   D&A 없음 (DART 요약 CF 에는 조정항목 없음) → |CapEx| − ΔPPE  (처분 무시, ≥0)
    if _missing("da_cf") and "ppe" in wide.columns and not _missing("capex_tangible"):
        _fill("da_cf", (wide["capex_tangible"].abs() - wide["ppe"].diff()).clip(lower=0), "← |CapEx| − ΔPPE (대체 추정)")
    if _missing("intangible_amort_cf") and "intangibles" in wide.columns and not _missing("capex_intangible"):
        _fill("intangible_amort_cf", (wide["capex_intangible"].abs() - wide["intangibles"].diff()).clip(lower=0), "← |CapEx무형| − Δ무형자산 (대체 추정)")

    mapping_df = pd.DataFrame(mapping_rows)
    if verbose:
        n_ok = mapping_df[mapping_df["match"] != "-"]["std_key"].nunique()
        log(ticker, f"DART wide shape={wide.shape}  {wide.index.min().date()}~{wide.index.max().date()}  "
                    f"표준키 {n_ok}/{len(DART_ACCOUNT_MAP)} 매핑  IS={is_mode} CF={cf_mode}")
    return wide, mapping_df


def show_dart_account_coverage(ticker: str, db_info: dict = None):
    """개별 종목 점검: 표준키별 채택 계정 + 최근 8분기 값 (억원)."""
    db_info = db_info or get_db_info()
    wide, mp = load_dart_financials_wide(ticker, db_info, verbose=True)
    if wide.empty:
        print(f"[{ticker}] DART 데이터 없음"); return None, None
    print("\n[표준키 매핑]")
    print(mp.to_string(index=False))
    print("\n[최근 8분기 (억원)]")
    print((wide.tail(8).T / 1e8).round(0).to_string())
    return wide, mp


def show_dart_is_accounts(ticker: str, sj: Tuple[str, ...] = ("IS", "CIS"), db_info: dict = None) -> pd.DataFrame:
    """매핑 실패 진단: 해당 종목의 계정 목록 — account_id / account_nm / 등장 횟수."""
    ticker = str(ticker).upper().lstrip("A").zfill(6)
    df = load_dart_long(ticker, db_info or get_db_info())
    out = (df[df["sj_div"].isin(list(sj))].groupby(["sj_div", "account_id", "account_nm"]).size()
           .reset_index(name="n").sort_values(["sj_div", "n"], ascending=[True, False]))
    print(out.to_string(index=False)); return out

## PART 2 · 매출 예측 (Korea_revenue_forecast_v4 준용) → `korea_revenue_forecast_from_DART`

- `_prepare_series` / `_run_models` 는 v4 와 동일 (최근 연속구간 추출, 전 모형 실패 차단, median 앙상블, n_models)
- 원천만 DART wide 의 `revenue` 로 교체 (단위 원)
- **저장 테이블 신규** — `forecast_date`(계측일) 가 UNIQUE KEY 에 포함되어 **계측일별 vintage** 가 누적됨 (요구사항 4)

In [6]:
def get_dart_revenue_series(ticker: str, db_info: dict, verbose: bool = False) -> Tuple[pd.Series, pd.DataFrame]:
    """DART wide → 분기 매출 PeriodIndex Series (원). v4 _prepare_series 와 동일 규칙."""
    wide, _ = load_dart_financials_wide(ticker, db_info, verbose=verbose)
    if wide.empty or "revenue" not in wide.columns:
        return pd.Series(dtype=float), wide
    rev = wide["revenue"].dropna()
    rev = rev[rev != 0]
    if rev.empty:
        return pd.Series(dtype=float), wide
    s = rev.copy()
    s.index = pd.PeriodIndex(s.index, freq="Q")
    s = s[~s.index.duplicated(keep="last")].sort_index().astype(float)

    # ── 최근 연속 구간 추출 (v4) ──
    full = pd.period_range(s.index.min(), s.index.max(), freq="Q")
    if len(full) != len(s):
        missing = full.difference(s.index)
        n_before = len(s)
        s = s.loc[s.index > missing.max()]
        if verbose:
            log(ticker, f"[결측분기 {len(missing)}개] {n_before}Q → 최근 연속 {len(s)}Q (마지막 결측 {missing.max()})")
    return s, wide


def _run_models(series: pd.Series, forecast_quarters: int) -> Tuple[pd.DataFrame, dict]:
    """v4 동일: SARIMA/ETS/Theta → wide DataFrame(index=예측 Period) + raw dict."""
    m = 4
    raw = {
        "SARIMA": forecast_sarima(y=series, forecast_horizon=forecast_quarters, seasonal_period=m, try_transforms=True),
        "ETS":    forecast_ets(y=series, forecast_horizon=forecast_quarters, m=m, try_transforms=True),
        "Theta":  forecast_theta(y=series, forecast_horizon=forecast_quarters, m=m, try_transforms=True),
    }
    ok = [k for k, r in raw.items() if isinstance(r, dict) and "forecast" in r]
    if not ok:
        errs = "; ".join(f"{k}:{(r or {}).get('error', 'unknown')}" for k, r in raw.items())
        raise ValueError(f"전 모형 적합 실패 ({errs})")
    periods = pd.period_range(start=series.index[-1] + 1, periods=forecast_quarters, freq="Q")
    fc = pd.DataFrame(index=periods)
    for name in ("SARIMA", "ETS", "Theta"):
        vals = raw[name].get("forecast") if isinstance(raw[name], dict) else None
        fc[name] = np.asarray(vals, dtype=float) if vals is not None else np.nan
    cols = ["SARIMA", "ETS", "Theta"]
    fc["Ensemble"] = fc[cols].median(axis=1) if ENSEMBLE_AGG == "median" else fc[cols].mean(axis=1)
    fc["n_models"] = fc[cols].notna().sum(axis=1)
    return fc, raw


def forecast_dart_revenue(ticker: str, db_info: dict,
                          forecast_quarters: int = FORECAST_QUARTERS,
                          min_data_periods: int = MIN_DATA_PERIODS,
                          verbose: bool = False) -> Tuple[bool, pd.DataFrame, str]:
    """
    단일 종목 예측. Returns (성공, wide[date,SARIMA,ETS,Theta,Ensemble,n_models,ticker,last_actual_date,n_obs], msg)
    """
    try:
        series, _ = get_dart_revenue_series(ticker, db_info, verbose=verbose)
        if series.empty:
            return False, pd.DataFrame(), "DART 매출 없음"
        if len(series) < min_data_periods:
            return False, pd.DataFrame(), f"데이터 부족 ({len(series)}Q < {min_data_periods}Q)"
        fc, _ = _run_models(series, forecast_quarters)
        fc["ticker"] = ticker
        fc["last_actual_date"] = series.index[-1].to_timestamp(how="end").normalize().date()
        fc["n_obs"] = len(series)
        fc.index = fc.index.to_timestamp(how="end").normalize()
        fc = fc.reset_index().rename(columns={"index": "date"})
        return True, fc, ""
    except Exception as e:
        if verbose:
            log(ticker, f"예측 실패 - {e}")
        return False, pd.DataFrame(), str(e)


# ── DB 저장 (vintage) ──────────────────────────────────────────
CREATE_DART_FC_SQL = f"""
CREATE TABLE IF NOT EXISTS {TABLE_DART_FC} (
    id               BIGINT AUTO_INCREMENT PRIMARY KEY,
    forecast_date    DATE        NOT NULL COMMENT '★ 계측일 (vintage key)',
    ticker           VARCHAR(20) NOT NULL COMMENT 'DART 6자리',
    date             DATE        NOT NULL COMMENT '예측 대상 분기말',
    indicator        VARCHAR(50) NOT NULL COMMENT 'SARIMA|ETS|Theta|Ensemble',
    value            DOUBLE               COMMENT '매출 (원)',
    n_models         TINYINT              COMMENT '앙상블 구성 모형 수',
    last_actual_date DATE                 COMMENT '예측 입력 마지막 실적 분기',
    n_obs            INT                  COMMENT '입력 분기 수',
    unit             VARCHAR(10) DEFAULT 'KRW',
    source           VARCHAR(40) DEFAULT '{TABLE_DART_FS}',
    ensemble_agg     VARCHAR(10),
    fc_module        VARCHAR(60),
    created_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    UNIQUE KEY uq_vintage (forecast_date, ticker, date, indicator),
    INDEX idx_ticker (ticker), INDEX idx_fdate (forecast_date), INDEX idx_date (date)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COMMENT='DART 기반 분기 매출 예측 (계측일별 vintage)'
"""

def ensure_dart_fc_table(db_info: dict):
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(CREATE_DART_FC_SQL)
        conn.commit()
    finally:
        conn.close()


def convert_to_long_format(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["date", "ticker", "indicator", "value", "n_models", "last_actual_date", "n_obs"])
    long_df = df.melt(id_vars=["date", "ticker", "n_models", "last_actual_date", "n_obs"],
                      value_vars=["SARIMA", "ETS", "Theta", "Ensemble"],
                      var_name="indicator", value_name="value")
    return long_df.sort_values(["ticker", "date", "indicator"]).reset_index(drop=True)


def save_dart_forecasts(forecasts_long: pd.DataFrame, db_info: dict,
                        forecast_date: date = FORECAST_DATE, batch_size: int = 200) -> int:
    """UNIQUE(forecast_date, ticker, date, indicator) → 같은 계측일 재실행 시 value 갱신."""
    if forecasts_long.empty:
        print("저장할 데이터 없음"); return 0
    ensure_dart_fc_table(db_info)
    sql = f"""
        INSERT INTO {TABLE_DART_FC}
            (forecast_date, ticker, date, indicator, value, n_models, last_actual_date, n_obs,
             unit, source, ensemble_agg, fc_module)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,'KRW',%s,%s,%s)
        ON DUPLICATE KEY UPDATE value=VALUES(value), n_models=VALUES(n_models),
            last_actual_date=VALUES(last_actual_date), n_obs=VALUES(n_obs),
            ensemble_agg=VALUES(ensemble_agg), fc_module=VALUES(fc_module), updated_at=CURRENT_TIMESTAMP
    """
    rows = [(forecast_date, r.ticker, pd.Timestamp(r.date).date(), r.indicator,
             float(r.value) if pd.notna(r.value) else None, int(r.n_models),
             r.last_actual_date, int(r.n_obs), TABLE_DART_FS, ENSEMBLE_AGG, FORECAST_MODULE)
            for r in forecasts_long.itertuples(index=False)]
    conn = get_pymysql_conn(db_info)
    n = 0
    try:
        with conn.cursor() as cur:
            for i in range(0, len(rows), batch_size):
                cur.executemany(sql, rows[i:i + batch_size])
                n += len(rows[i:i + batch_size])
        conn.commit()
    except Exception:
        conn.rollback(); raise
    finally:
        conn.close()
    print(f"  저장 완료 | {n:,}행 → {TABLE_DART_FC}  (forecast_date={forecast_date})")
    return n


def forecast_exists(ticker: str, db_info: dict, forecast_date: date) -> bool:
    ensure_dart_fc_table(db_info)
    with engine.connect() as c:
        r = c.execute(_sa_text(f"SELECT COUNT(*) FROM {TABLE_DART_FC} WHERE ticker=:t AND forecast_date=:d"),
                      {"t": ticker, "d": forecast_date}).fetchone()
    return bool(r and r[0] > 0)


def run_dart_forecast_batch(tickers: List[str], db_info: dict, forecast_date: date = FORECAST_DATE,
                            rerun: bool = RERUN_FORECAST, verbose: bool = False) -> Dict[str, Any]:
    """다수 종목 예측 + 저장. (v4 process_all_tickers 축약판)"""
    ok, fail, buf = 0, [], []
    t0 = time.time()
    for tk in tqdm(tickers, desc="DART 매출예측"):
        if not rerun and forecast_exists(tk, db_info, forecast_date):
            if verbose: log(tk, f"forecast_date={forecast_date} 이미 존재 → skip")
            ok += 1; continue
        s, df, msg = forecast_dart_revenue(tk, db_info, verbose=verbose)
        if s:
            ok += 1; buf.append(df)
        else:
            fail.append((tk, msg))
        gc.collect()
    if buf:
        save_dart_forecasts(convert_to_long_format(pd.concat(buf, ignore_index=True)), db_info, forecast_date)
    print(f"[예측 완료] 성공 {ok} / 실패 {len(fail)}  경과 {time.time()-t0:.0f}s")
    for tk, m in fail:
        print(f"  ✗ {tk}: {m}")
    return {"success": ok, "fail": fail}


# ── 조회: 최신(또는 지정) vintage 로드 + v1 3중 검증 ──────────
_FC_MODEL_PRIORITY = ("Ensemble", "SARIMA", "ETS", "Theta")

def load_dart_revenue_forecast(ticker: str, db_info: dict, horizon: int = FORECAST_HORIZON,
                               forecast_date: Optional[date] = None) -> Tuple[pd.Series, str, Optional[str]]:
    """
    Returns (forecast Series[원, index=분기말], indicator, forecast_date str)
    forecast_date=None → 가장 최근 계측일.  v1 패치의 3중 검증(분기수·연속성·상수) 동일 적용.
    """
    ensure_dart_fc_table(db_info)
    with engine.connect() as c:
        if forecast_date is None:
            r = c.execute(_sa_text(f"SELECT MAX(forecast_date) FROM {TABLE_DART_FC} WHERE ticker=:t"), {"t": ticker}).fetchone()
            if r is None or r[0] is None:
                return pd.Series(dtype=float), "", None
            forecast_date = r[0]
    fd = str(forecast_date)
    for model in _FC_MODEL_PRIORITY:
        df = pd.read_sql(_sa_text(f"SELECT date, value FROM {TABLE_DART_FC} "
                                  f"WHERE ticker=:t AND forecast_date=:d AND indicator=:m ORDER BY date"),
                         engine, params={"t": ticker, "d": fd, "m": model})
        df = df.dropna(subset=["value"])
        if not df.empty:
            break
    else:
        return pd.Series(dtype=float), "", fd
    fc = df.assign(date=pd.to_datetime(df["date"])).set_index("date")["value"].astype(float).sort_index()
    if len(fc) < horizon:
        raise ValueError(f"[{ticker}] forecast {len(fc)}Q < horizon {horizon} (vintage {fd}) → 재예측 필요")
    fc = fc.iloc[:horizon]
    per = fc.index.to_period("Q")
    if ((per[1:].astype("int64") - per[:-1].astype("int64")) != 1).any():
        raise ValueError(f"[{ticker}] forecast 분기 불연속 (vintage {fd})")
    if fc.nunique() == 1:
        raise ValueError(f"[{ticker}] forecast 전 분기 동일값 — 상수 시계열 의심 (vintage {fd})")
    return fc, model, fd


def load_forecast_vintages(ticker: str, db_info: dict, indicator: str = "Ensemble") -> pd.DataFrame:
    """계측일(forecast_date) × 대상분기(date) 피벗 — 예측이 계측일별로 어떻게 변했는지 추적 (요구사항 4)."""
    ensure_dart_fc_table(db_info)
    df = pd.read_sql(_sa_text(f"SELECT forecast_date, date, value, last_actual_date FROM {TABLE_DART_FC} "
                              f"WHERE ticker=:t AND indicator=:m ORDER BY forecast_date, date"),
                     engine, params={"t": ticker, "m": indicator})
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"]).dt.to_period("Q").astype(str)
    pv = df.pivot_table(index="forecast_date", columns="date", values="value", aggfunc="last")
    last = df.groupby("forecast_date")["last_actual_date"].first()
    pv.insert(0, "last_actual", last.astype(str))
    return pv


def inspect_dart_ticker(ticker: str, db_info: dict = None, tail_n: int = 20, forecast_quarters: int = FORECAST_QUARTERS) -> dict:
    """개별 종목 점검 (DB 저장 X): 입력 매출 tail + 모델별 예측 + 적합 상태 (v4 inspect_ticker 준용)."""
    db_info = db_info or get_db_info()
    ticker = str(ticker).upper().lstrip("A").zfill(6)
    series, wide = get_dart_revenue_series(ticker, db_info, verbose=True)
    if series.empty:
        print(f"[{ticker}] DART 매출 없음"); return {}
    fc, raw = _run_models(series, forecast_quarters)
    tail = (series.tail(tail_n) / 1e8).rename("revenue(억원)").to_frame(); tail.index = tail.index.astype(str)
    print("=" * 70); print(f" {ticker} | 사용 {len(series)}Q | 예측 {forecast_quarters}Q | 앙상블 {ENSEMBLE_AGG}"); print("=" * 70)
    print(tail.to_string(float_format=lambda x: f"{x:,.0f}"))
    print("\n[모델 적합 상태]")
    for nm in ("SARIMA", "ETS", "Theta"):
        r = raw.get(nm, {}) or {}
        if "forecast" in r:
            spec = r.get("spec", {}) or {}
            extra = f"  order={spec.get('order')} seasonal={spec.get('seasonal_order')}" if nm == "SARIMA" else ""
            print(f"   {nm:<7} 성공   transform={','.join(r.get('used_transform', [])) or 'none'}{extra}")
        else:
            print(f"   {nm:<7} 실패   {r.get('error', 'unknown')}")
    disp = (fc[["SARIMA", "ETS", "Theta", "Ensemble"]] / 1e8).round(0); disp["n_models"] = fc["n_models"]; disp.index = disp.index.astype(str)
    print("\n[모델별 예측 (억원)]"); print(disp.to_string(float_format=lambda x: f"{x:,.0f}"))
    return {"ticker": ticker, "input_series": series, "forecast": fc, "raw": raw, "wide": wide}

## PART 3 · `DartRIMModel` — KoreaRIMModel v5 (선형 fade) 그대로, 데이터 로더만 DART

`IV = BV₀ + Σ PV(RI_t)` , `RI_t = (ROE_t − Re) × BV_{t−1}` , Phase 2 종료 시 spread→0 ⇒ **TV = 0** (v5 구조).
변경점은 `load_sales` / `load_financials` / `_estimate_shares` 뿐. DuPont 계수·Phase 1 blend/floor·6-Tier moat·n_pos 페널티·
선형 fade 는 `korea_rim_valuation_v5` (Cell 5 + Cell 5-B 패치) 원문과 동일.

| DART 로 추정되는 항목 | 방식 |
|---|---|
| 주식수 | `시가총액 ÷ 현재가` |
| 배당(retention) | CF `배당금지급` 계정 → 없으면 Clean Surplus 역산 `NI − ΔEquity` (v5 원문 fallback) |
| 이자비용 (EVA 용 WACC) | `이자비용` 없으면 `금융비용` |
| net_income | `ifrs-full_ProfitLoss` (지배+비지배 합계 — total_equity 와 정합) |

In [7]:
class DartRIMModel:
    """DART 원천 Sales-driven DuPont ROE → Ohlson RIM (KoreaRIMModel v5 이식)."""

    def __init__(self, ticker, engine, db_info, rf, e_rm, kospi_series,
                 forecast_horizon=FORECAST_HORIZON, min_history=MIN_HISTORY, gdp_growth=GDP_GROWTH,
                 forecast_date: Optional[date] = None, verbose=False):
        self.ticker_dart  = str(ticker).upper().lstrip("A").zfill(6)
        self.ticker_dg    = to_dg_ticker(self.ticker_dart)
        self.ticker_price = to_price_ticker(self.ticker_dart)
        self.engine, self.db_info = engine, db_info
        self.rf, self.e_rm, self.erp = rf, e_rm, e_rm - rf
        self.kospi = kospi_series
        self.horizon, self.min_history, self.gdp_growth = forecast_horizon, min_history, gdp_growth
        self.forecast_date_req = forecast_date
        self.verbose = verbose
        self._sales_actual = None; self._sales_forecast = None
        self._used_model = ""; self._forecast_date = None
        self._fs_wide = None; self._mapping_df = None
        self._re = None; self._eva_cache = None; self._beta_info = None
        self._using_ic = False; self._n_phase1 = 2; self._bv_source = "Equity"
        self._shares_method = "?"; self._coefs = {}
        self.result_df = None; self.valuation = None
        self.report = DataQualityReport(ticker=self.ticker_dart)

    # ── Utility (v5 원문) ─────────────────────────────────────
    @staticmethod
    def _winsorize(s, limits=WINSORIZE_LIMITS):
        s = s.dropna()
        if len(s) < 4: return s
        return s.clip(s.quantile(limits[0]), s.quantile(limits[1]))

    @staticmethod
    def _ols_ratio(x, y):
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES: return np.nan, -1.0, int(mask.sum())
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return float(slope), float(r ** 2), int(mask.sum())

    # ── 1. 데이터 로드 (★ DART) ──────────────────────────────
    def load_sales(self):
        series, _ = get_dart_revenue_series(self.ticker_dart, self.db_info, verbose=self.verbose)
        if series.empty:
            self.report.add("revenue_actual", "missing", n_obs=0, note="DART 매출 없음")
            raise ValueError(f"[{self.ticker_dart}] DART Sales actual 없음")
        actual = series.copy(); actual.index = actual.index.to_timestamp(how="end").normalize()
        actual = actual * FS_UNIT_MULTIPLIER
        if len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual), note=f"actual {len(actual)}Q < {MIN_REVENUE_QUARTERS}")
            raise ValueError(f"[{self.ticker_dart}] 매출 actual 부족 ({len(actual)}Q < {MIN_REVENUE_QUARTERS}Q)")
        self.report.add("revenue_actual", "ok", n_obs=len(actual))
        forecast, model_name, fd = load_dart_revenue_forecast(self.ticker_dart, self.db_info, horizon=self.horizon,
                                                              forecast_date=self.forecast_date_req)
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0, note=f"{TABLE_DART_FC} 에 예측 없음")
            raise ValueError(f"[{self.ticker_dart}] 매출 forecast 없음 — PART 2 예측 먼저 실행")
        forecast = forecast * FS_UNIT_MULTIPLIER
        exp_start = actual.index[-1].to_period("Q") + 1
        if forecast.index[0].to_period("Q") != exp_start:
            self.report.warn(f"forecast 시작 {forecast.index[0].to_period('Q')} ≠ 실적 다음 분기 {exp_start} (vintage {fd})")
        self.report.add("revenue_forecast", "ok", n_obs=len(forecast), note=f"model={model_name}, forecast_date={fd}")
        self._sales_actual, self._sales_forecast = actual, forecast.iloc[:self.horizon]
        self._used_model, self._forecast_date = model_name, fd
        if self.verbose: log(self.ticker_dart, f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q ({model_name}, {fd})")
        return self

    def load_financials(self):
        wide, mp = load_dart_financials_wide(self.ticker_dart, self.db_info, verbose=self.verbose)
        if wide.empty:
            raise ValueError(f"[{self.ticker_dart}] DART wide 비어있음")
        wide = wide * FS_UNIT_MULTIPLIER
        missing = [k for k in ("revenue", "net_income", "total_equity") if k not in wide.columns or wide[k].dropna().empty]
        if missing:
            for k in missing: self.report.add(k, "missing", n_obs=0)
            raise ValueError(f"[{self.ticker_dart}] 핵심 항목 결측: {missing}")
        self.report.add("fs_core", "ok", n_obs=int(wide[["revenue", "net_income", "total_equity"]].notna().all(axis=1).sum()))
        self._fs_wide, self._mapping_df = wide, mp
        return self

    # ── 2. Re ─────────────────────────────────────────────────
    def load_re(self):
        info = compute_beta_10y(self.ticker_dg, self.db_info, kospi_series=self.kospi, years=10, min_obs=750)
        self._beta_info = info
        if np.isnan(info["beta_raw"]):
            bb = 1.0; self.report.add("beta", "fallback_median", n_obs=info["n_obs"], value=bb, note="베타 실패 → 1.0")
        else:
            bb = info["beta_blume"]; self.report.add("beta", "ok", n_obs=info["n_obs"], value=bb, r2=info["r_squared"], note=f"β_raw={info['beta_raw']:.3f}")
        self._re = float(self.rf + bb * self.erp)     # v2 옵션 A: clip 없음
        if self.verbose: log(self.ticker_dart, f"Re={self._re:.3%} β_raw={info.get('beta_raw', np.nan):.3f} β_blume={bb:.3f}")
        return self

    # ── 3. 보조 (v5 원문) ─────────────────────────────────────
    def _total_debt_series(self):
        wide = self._fs_wide; td = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns: td = td + wide[k].fillna(0)
        return td

    def _estimate_ic(self):
        wide = self._fs_wide.sort_index(); last = wide.iloc[-1]
        eq = last.get("total_equity", 0) or 0
        td = sum(float(last.get(k, 0)) for k in DEBT_KEYS if k in wide.columns and pd.notna(last.get(k, np.nan)))
        ic = float(eq) + td
        if ic <= 0:
            ta = last.get("total_assets", 0); ic = float(ta) * 0.40 if pd.notna(ta) and ta > 0 else 1e11
        return max(ic, 1e10)

    def _get_historical_roe_ttm(self):
        try:
            df = self._fs_wide.sort_index()[["net_income", "total_equity"]].dropna(); df = df[df["total_equity"] > 0]
            if len(df) < 4: return np.nan
            eq_avg = float(df["total_equity"].iloc[-4:].mean())
            return float(df["net_income"].iloc[-4:].sum()) / eq_avg if eq_avg > 0 else np.nan
        except Exception:
            return np.nan

    def estimate_tax_rate(self):
        wide = self._fs_wide
        if "pretax_income" not in wide.columns or "tax_expense" not in wide.columns:
            self.report.add("tax_rate", "fallback_zero", n_obs=0, value=0.22, note="컬럼 누락 → 22%"); return 0.22
        df = wide[["pretax_income", "tax_expense"]].dropna(); df = df[df["pretax_income"] > 0]
        if df.empty:
            self.report.add("tax_rate", "fallback_zero", n_obs=0, value=0.22, note="법정세율 22%"); return 0.22
        med = float((df["tax_expense"] / df["pretax_income"]).clip(0, 0.40).median())
        self.report.add("tax_rate", "ok", n_obs=len(df), value=med); return med

    def estimate_retention(self):
        wide = self._fs_wide.sort_index()
        ni = wide["net_income"]
        if (ni > 0).sum() == 0:
            self.report.add("retention", "fallback_zero", value=0.70, note="NI 양수 분기 0"); return 0.70
        div, src = None, ""
        if "dividends_paid" in wide.columns:
            d = wide["dividends_paid"].abs()
            if (d > 0).sum() >= 4: div, src = d, "DART CF 배당금지급"
        if div is None and "total_equity" in wide.columns:
            implied = (ni - wide["total_equity"].diff()).clip(lower=0)
            if (implied > 0).sum() >= 4: div, src = implied, "clean-surplus 역산"
        if div is None:
            self.report.add("retention", "fallback_zero", value=0.70, note="배당/clean surplus 모두 실패"); return 0.70
        payout = (div / ni.abs()).replace([np.inf, -np.inf], np.nan)
        po = payout.iloc[-8:].dropna()
        if po.empty: po = payout.dropna()
        if po.empty:
            self.report.add("retention", "fallback_zero", value=0.70); return 0.70
        payout_med = float(po.clip(0, 3.0).median())
        retention = float(np.clip(1.0 - payout_med, RETENTION_FLOOR, 0.98))
        self.report.add("retention", "ok", n_obs=len(po), value=retention, note=f"payout_med={payout_med:.2%} ({src})")
        if self.verbose: log(self.ticker_dart, f"Retention={retention:.3f} payout={payout_med:.3f} [{src}]")
        return retention

    # ── 4. DuPont (v5 원문) ───────────────────────────────────
    def estimate_dupont_coefs(self):
        wide = self._fs_wide
        df_npm = wide[["revenue", "net_income"]].dropna(); df_npm = df_npm[df_npm["revenue"] > 0]
        npm_series = self._winsorize((df_npm["net_income"] / df_npm["revenue"]).dropna())
        npm_median = float(npm_series.median()) if not npm_series.empty else 0.05
        npm_coef, npm_method, r2_npm = npm_median, "median", -1.0
        if len(df_npm) >= OLS_MIN_SAMPLES:
            slope, r2_npm, n = self._ols_ratio(df_npm["revenue"], df_npm["net_income"])
            if not np.isnan(slope) and r2_npm >= OLS_MIN_R2:
                npm_coef, npm_method = float(slope), "ols"
                self.report.add("npm", "ok", n_obs=n, value=npm_coef, r2=r2_npm, note="OLS slope")
            else:
                self.report.add("npm", "fallback_median", n_obs=n, value=npm_median, r2=r2_npm, note=f"R²={r2_npm:.2f} → median")
        else:
            self.report.add("npm", "fallback_median", n_obs=len(df_npm), value=npm_median, note=f"n={len(df_npm)} < {OLS_MIN_SAMPLES}")
        at_median = 0.70
        if "total_assets" in wide.columns:
            df_at = wide[["revenue", "total_assets"]].dropna().sort_index(); df_at = df_at[df_at["total_assets"] > 0]
            if len(df_at) >= 4:
                df_at["ttm"] = df_at["revenue"].rolling(4).sum(); atm = df_at.dropna(subset=["ttm"])
                ratios = self._winsorize((atm["ttm"] / atm["total_assets"]).dropna())
                if not ratios.empty:
                    at_median = float(ratios.median()); self.report.add("asset_turnover", "ok", n_obs=len(ratios), value=at_median)
                else: self.report.add("asset_turnover", "fallback_zero", value=at_median)
            else: self.report.add("asset_turnover", "fallback_zero", value=at_median, note="total_assets < 4Q")
        else: self.report.add("asset_turnover", "fallback_zero", value=at_median, note="total_assets 없음")
        fl_median = 2.5
        if "total_assets" in wide.columns:
            df_fl = wide[["total_assets", "total_equity"]].dropna(); df_fl = df_fl[(df_fl["total_assets"] > 0) & (df_fl["total_equity"] > 0)]
            if not df_fl.empty:
                ratios = self._winsorize((df_fl["total_assets"] / df_fl["total_equity"]).dropna())
                fl_median = float(np.clip(ratios.median(), 1.0, 20.0)); self.report.add("financial_leverage", "ok", n_obs=len(ratios), value=fl_median)
            else: self.report.add("financial_leverage", "fallback_zero", value=fl_median, note="equity>0 분기 없음")
        else: self.report.add("financial_leverage", "fallback_zero", value=fl_median)
        if self.verbose: log(self.ticker_dart, f"DuPont NPM={npm_coef:.4f}({npm_method}) AT={at_median:.3f} FL={fl_median:.2f}")
        return {"npm_coef": npm_coef, "npm_method": npm_method, "npm_r2": r2_npm, "at_median": at_median, "fl_median": fl_median}

    # ── 5. Phase 1 ROE (v5 원문) ──────────────────────────────
    def forecast_roe_phase1(self, coefs, n_years=2):
        fc_q, re = self._sales_forecast, self._re
        annual_raw = [{"year": fc_q.index[yr].year, "sales": float(fc_q.iloc[yr:yr+4].sum()), "source": "forecast"}
                      for yr in range(0, len(fc_q), 4)]
        last_yr = annual_raw[-1]["year"] if annual_raw else datetime.now().year
        last_sal = annual_raw[-1]["sales"] if annual_raw else 0.0
        annual = list(annual_raw)
        for i in range(len(annual_raw), n_years):
            extra = i - len(annual_raw) + 1
            annual.append({"year": last_yr + extra, "sales": last_sal * (1 + GDP_GROWTH) ** extra, "source": "gdp_ext"})
        eq_ser = self._fs_wide.sort_index()["total_equity"].dropna()
        bv0_raw = float(eq_ser.iloc[-1]) if not eq_ser.empty else -1.0
        if bv0_raw <= 0:
            bv0 = self._estimate_ic(); self._using_ic, self._bv_source = True, "IC_fallback"
            self.report.add("bv0", "fallback_median", n_obs=0, value=bv0, note=f"음수 BV({bv0_raw/1e12:.2f}조) → IC")
        else:
            bv0 = bv0_raw; self._using_ic, self._bv_source = False, "Equity"; self.report.add("bv0", "ok", value=bv0)
        retention = self.estimate_retention(); hist_roe = self._get_historical_roe_ttm()
        rows, bv_start, roe_fixed = [], bv0, None
        for yi in annual:
            is_flat = yi.get("source") == "gdp_ext"
            sales = yi["sales"]; ni = coefs["npm_coef"] * sales; npm = ni / sales if sales > 0 else 0.0
            at_est, fl_est = coefs["at_median"], coefs["fl_median"]
            if not is_flat:
                roe_raw = npm * at_est * fl_est
                if not np.isnan(hist_roe) and hist_roe > 0:
                    roe_blend = 0.65 * roe_raw + 0.35 * hist_roe
                    floor_roe, ceil_roe = hist_roe * PHASE1_ROE_FLOOR_RATIO, hist_roe * PHASE1_ROE_CEILING_RATIO
                    if roe_raw < floor_roe: roe = max(roe_blend, floor_roe)
                    elif roe_raw > ceil_roe: roe = min(roe_blend, hist_roe * 1.50)
                    else: roe = roe_blend
                    roe = float(np.clip(roe, -0.99, 3.0))
                else:
                    roe = float(np.clip(roe_raw, -0.99, 2.5))
                roe_fixed = roe
            else:
                roe = roe_fixed if roe_fixed is not None else (hist_roe if not np.isnan(hist_roe) else re + 0.05)
            ri_spread = roe - re; ri = ri_spread * bv_start
            rows.append({"year": yi["year"], "phase": "ph1" if not is_flat else "ph1e", "sales_annual": sales, "net_income": ni,
                         "npm": npm, "at": at_est, "fl": fl_est, "roe": roe, "re": re, "ri_spread": ri_spread, "bv_start": bv_start, "ri": ri})
            b_bv = retention if retention < 0 else min(retention, BV_RETENTION_CAP)
            bv_start = bv_start * (1.0 + re * b_bv) + ri * b_bv
        return pd.DataFrame(rows), bv_start

    # ── 6. EVA / Moat 6-Tier (v5 원문) ───────────────────────
    def compute_wacc_simple(self):
        re = self._re or 0.10; tax = self.estimate_tax_rate(); wide = self._fs_wide
        rd = RD_DEFAULT
        if "interest_expense" in wide.columns:
            td = self._total_debt_series(); td_avg = (td + td.shift(1)) / 2; ie = wide["interest_expense"].abs()
            valid = (td_avg > 0) & ie.notna()
            if valid.sum() >= 2:
                mult = 4.0 if RD_ANNUALIZE_QUARTERLY else 1.0
                rd = float((ie[valid] * mult / td_avg[valid]).clip(0, 0.20).median())
        mc, _ = load_korea_marketcap_latest(self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        if mc is None or mc <= 0: return re
        E = mc * MARKETCAP_UNIT_MULTIPLIER
        tdl = sum(float(wide[k].dropna().iloc[-1]) for k in DEBT_KEYS if k in wide.columns and not wide[k].dropna().empty)
        V = E + tdl
        if V <= 0: return re
        return float(np.clip(re * E / V + rd * (1 - tax) * tdl / V, 0.04, 0.25))

    def compute_eva_spread(self):
        wacc = self.compute_wacc_simple(); tax = self.estimate_tax_rate(); wide = self._fs_wide
        empty = {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan, "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}
        if "operating_income" not in wide.columns: return empty
        nopat_q = wide["operating_income"] * (1 - tax)
        cash = pd.Series(0.0, index=wide.index)
        for k in CASH_KEYS:
            if k in wide.columns: cash = cash + wide[k].fillna(0)
        ic_s = wide["total_equity"].fillna(0) + self._total_debt_series() - cash
        merged = pd.concat([nopat_q.rename("nopat_q"), ic_s.rename("ic")], axis=1, join="inner").dropna().sort_index()
        if len(merged) < 4: return empty
        eva_series = []
        for i in range(3, len(merged)):
            ic = float(merged["ic"].iloc[i])
            if ic <= 0: continue
            roic = float(merged["nopat_q"].iloc[i-3:i+1].sum()) / ic
            eva_series.append({"date": merged.index[i], "roic": roic, "eva_spread": roic - wacc})
        if not eva_series: return empty
        latest = eva_series[-1]; n_pos = sum(1 for e in eva_series[-20:] if e["eva_spread"] > 0)
        re = self._re or 0.10; roe_re = np.nan
        df_re = wide[["net_income", "total_equity"]].dropna(); df_re = df_re[df_re["total_equity"] > 0]
        if len(df_re) >= 4:
            eq_avg = float(df_re["total_equity"].iloc[-4:].mean())
            if eq_avg > 0: roe_re = float(df_re["net_income"].iloc[-4:].sum() / eq_avg - re)
        res = {"roic": latest["roic"], "wacc": wacc, "eva_spread": latest["eva_spread"], "roe_re_spread": roe_re, "n_positive": n_pos, "eva_series": eva_series}
        self._eva_cache = res
        if self.verbose:
            log(self.ticker_dart, f"EVA: ROIC={latest['roic']:.2%} WACC={wacc:.2%} spread={latest['eva_spread']:+.2%} n_pos={n_pos}/20 ROE-Re={'N/A' if np.isnan(roe_re) else f'{roe_re:+.1%}'}")
        return res

    def moat_to_omega_and_years(self, eva):
        eva_spread, roe_re, n_pos = eva.get("eva_spread", np.nan), eva.get("roe_re_spread", np.nan), eva.get("n_positive", 0)
        if np.isnan(eva_spread) and np.isnan(roe_re):
            return 0.760, 8, "Unknown (fallback)", PHASE1_YR_BY_MOAT["Unknown (fallback)"]
        eva_s = eva_spread if not np.isnan(eva_spread) else -np.inf
        roe_s = roe_re if not np.isnan(roe_re) else -np.inf
        GRADE = [("Exceptional moat", 0.990, 30, 0.25, 1.50), ("Wide moat", 0.980, 28, 0.15, 0.80),
                 ("Wide-Narrow moat", 0.970, 25, 0.10, 0.40), ("Narrow moat", 0.950, 20, 0.06, 0.20), ("Some moat", 0.940, 15, 0.03, 0.05)]
        grade, omega, n = "No moat", 0.880, 10
        for label, w, yr, eth, rth in GRADE:
            if eva_s > eth or roe_s > rth: grade, omega, n = label, w, yr; break
        N_POS_MIN = {"Exceptional moat": 18, "Wide moat": 16, "Wide-Narrow moat": 13, "Narrow moat": 10, "Some moat": 7, "No moat": 0}
        ORDER = ["Exceptional moat", "Wide moat", "Wide-Narrow moat", "Narrow moat", "Some moat", "No moat"]
        ALL = {"Exceptional moat": (0.990, 30), "Wide moat": (0.980, 28), "Wide-Narrow moat": (0.970, 25), "Narrow moat": (0.950, 20), "Some moat": (0.940, 15), "No moat": (0.880, 10)}
        if n_pos < N_POS_MIN.get(grade, 0):
            idx = ORDER.index(grade)
            if idx < len(ORDER) - 1: grade = ORDER[idx + 1]; omega, n = ALL[grade]
        return omega, n, grade, PHASE1_YR_BY_MOAT.get(grade, 2)

    # ── 7. RI Path — ★ v5 선형 fade (Cell 5-B 원문) ─────────
    def compute_ri_path(self):
        re = self._re
        coefs = self.estimate_dupont_coefs(); self._coefs = coefs
        eva = self.compute_eva_spread()
        omega, n_phase2, moat_label, n_phase1 = self.moat_to_omega_and_years(eva)
        ph1_df, bv_after_ph1 = self.forecast_roe_phase1(coefs, n_years=n_phase1)
        retention = self.estimate_retention(); self._retention = retention
        rows = ph1_df.to_dict("records")
        spread0 = float(ph1_df["ri_spread"].iloc[-1]) if not ph1_df.empty else 0.0
        bv_current = bv_after_ph1
        last_year = int(ph1_df["year"].iloc[-1]) if not ph1_df.empty else datetime.now().year
        g_term = float(np.clip(self.gdp_growth, 0.0, max(re - 0.005, 0.0)))
        b_term = g_term / re if re > 1e-6 else 0.0
        b0 = retention if retention < 0 else min(retention, BV_RETENTION_CAP)
        ri_t = float(ph1_df["ri"].iloc[-1]) if not ph1_df.empty else 0.0
        for t in range(1, n_phase2 + 1):
            f = t / float(n_phase2)
            spread_t = spread0 * (1.0 - f); b_t = b0 + (b_term - b0) * f
            if bv_current <= 0: bv_current, spread_t = 0.0, 0.0
            ri_t = spread_t * bv_current
            rows.append({"year": last_year + t, "phase": "ph2", "sales_annual": np.nan, "net_income": np.nan, "npm": np.nan, "at": np.nan, "fl": np.nan,
                         "roe": re + spread_t, "re": re, "ri_spread": spread_t, "bv_start": bv_current, "ri": ri_t})
            bv_current = bv_current * (1.0 + re * b_t) + ri_t * b_t
        self.result_df = pd.DataFrame(rows)
        self._bv_terminal, self._ri_last, self._omega = bv_current, ri_t, omega
        self._moat_label, self._n_phase2, self._n_phase1, self._eva = moat_label, n_phase2, len(ph1_df), eva
        self._g_term_fade, self._b_term = g_term, b_term
        if self.verbose:
            log(self.ticker_dart, f"RI path(LINEAR-fade v5): Ph1={len(ph1_df)}yr Ph2={n_phase2}yr spread {spread0:.1%}→0  b {b0:.2f}→{b_term:.2f}  RI_last={ri_t/1e9:,.1f}B → TV=0 [{moat_label}]")
        return self

    # ── 8. Valuation (v5 원문 + 주식수 DART 추정) ─────────────
    def _estimate_shares(self):
        mc, mc_date = load_korea_marketcap_latest(self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        if mc and mc > 0 and cp and cp > 0:
            sh = mc * MARKETCAP_UNIT_MULTIPLIER / cp; self._shares_method = f"marketcap/price ({mc_date})"
            self.report.add("shares", "ok", value=sh, note=self._shares_method); return float(sh)
        self._shares_method = "missing"; self.report.add("shares", "missing", n_obs=0); return np.nan

    def compute_valuation(self):
        re = self._re; df = self.result_df.copy()
        eq_ser = self._fs_wide.sort_index()["total_equity"].dropna()
        bv0_raw = float(eq_ser.iloc[-1]) if not eq_ser.empty else -1.0
        bv0 = bv0_raw if bv0_raw > 0 else self._estimate_ic()
        df["pv_ri"] = [row["ri"] / (1 + re) ** t for t, (_, row) in enumerate(df.iterrows(), 1)]
        pv_ri_total = float(df["pv_ri"].sum()); T = len(df)
        ri_last = self._ri_last
        g_tv = min(max(re - TV_RE_GAP, self.gdp_growth), re - 0.005)
        tv = 0.0
        if ri_last > 0 and re > g_tv: tv = ri_last * (1.0 + g_tv) / (re - g_tv)
        elif ri_last > 0: tv = ri_last * (1.0 + g_tv) / 0.005
        pv_tv = tv / (1 + re) ** T if tv != 0 else 0.0
        intrinsic = bv0 + pv_ri_total + pv_tv
        shares = self._estimate_shares()
        target_price = intrinsic / shares if (not np.isnan(shares) and shares > 0) else np.nan
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        if cp is None: self.report.add("current_price", "missing", n_obs=0); cp = np.nan
        upside = ((target_price / cp) - 1) * 100 if (not np.isnan(target_price) and not np.isnan(cp) and cp > 0) else np.nan
        self.result_df = df
        self.valuation = {
            "ticker": self.ticker_dart, "re": re, "g_terminal": g_tv, "g_term_fade": self._g_term_fade, "b_term": self._b_term,
            "bv0": bv0, "bv_source": self._bv_source, "pv_ri": pv_ri_total, "terminal_value": tv, "pv_tv": pv_tv,
            "tv_weight_pct": pv_tv / intrinsic * 100 if intrinsic else np.nan, "intrinsic_value": intrinsic,
            "shares": shares, "shares_method": self._shares_method, "target_price": target_price, "current_price": cp, "upside_pct": upside,
            "moat_label": self._moat_label, "omega": self._omega, "n_phase1": self._n_phase1, "n_phase2": self._n_phase2,
            "retention": self._retention, "eva_spread": self._eva.get("eva_spread", np.nan), "roic": self._eva.get("roic", np.nan),
            "wacc_eva": self._eva.get("wacc", np.nan), "roe_re_spread": self._eva.get("roe_re_spread", np.nan), "n_positive": self._eva.get("n_positive", 0),
            "beta_raw": self._beta_info.get("beta_raw") if self._beta_info else np.nan,
            "beta_blume": self._beta_info.get("beta_blume") if self._beta_info else np.nan,
            "hist_roe_ttm": self._get_historical_roe_ttm(),
            "revenue_quarters": len(self._sales_actual), "forecast_model": self._used_model, "forecast_date": self._forecast_date,
            "data_source": TABLE_DART_FS,
        }
        if self.verbose:
            tp = f"{target_price:,.0f}원" if not np.isnan(target_price) else "N/A"; cps = f"{cp:,.0f}원" if not np.isnan(cp) else "N/A"
            up = f"{upside:+.1f}%" if not np.isnan(upside) else "N/A"
            log(self.ticker_dart, f"RIM BV={bv0/1e12:.2f}조 PV(RI)={pv_ri_total/1e12:.2f}조 PV(TV)={pv_tv/1e12:.2f}조 IV={intrinsic/1e12:.2f}조 TP={tp} CP={cps} Up={up} [{self._moat_label}]")
        return self

    def run(self):
        self.load_sales(); self.load_financials(); self.load_re(); self.compute_ri_path(); self.compute_valuation(); return self


print("[OK] DartRIMModel 정의 완료 (v5 선형 fade 내장)")

[OK] DartRIMModel 정의 완료 (v5 선형 fade 내장)


## PART 4 · Excel 출력 (DB 저장 없음)

파일명: `{ticker}_{기업명}_RIM_DART_{YYYYMMDD}.xlsx` → `EXPORT_DIR` (FCFF 파일과 같은 폴더, 태그로 구분)

| 시트 | 내용 |
|---|---|
| Summary | 적정주가·IV 분해(BV₀ / ΣPV(RI) / PV(TV))·Re·Moat·ω·Phase 기간·retention·추정 방식 |
| RI_Schedule | Phase 1(forecast/plateau) + Phase 2(선형 fade) 연도별 ROE / spread / BV / RI / PV(RI) |
| DuPont_Q | 분기별 NPM × AT × FL → ROE (최근 40Q) + Phase 1 적용 계수 |
| Moat_Tiers | 6-Tier 기준표 + 현재 등급 |
| Sales | 실적 + 예측 매출 (모델별) |
| Forecast_Vintages | 계측일별 Ensemble 예측 변화 |
| Financials_Q | DART 분기 wide (억원) |
| Account_Mapping / Quality / Assumptions | 매핑 · 품질 · 파라미터 |

In [8]:
def _corp_name(ticker: str) -> str:
    if ticker in CORP_NAMES:
        return CORP_NAMES[ticker]
    for col_n in ("name", "corp_name", "company_name"):
        try:
            with engine.connect() as c:
                r = c.execute(_sa_text(f"SELECT {col_n} FROM {TABLE_MARKETCAP} WHERE ticker=:t ORDER BY date DESC LIMIT 1"),
                              {"t": to_dg_ticker(ticker)}).fetchone()
            if r and r[0]:
                return re.sub(r"[\\/:*?\"<>|\s]+", "_", str(r[0]))
        except Exception:
            continue
    return "NA"


def _quality_df(report) -> pd.DataFrame:
    for attr in ("to_dataframe", "to_df", "as_dataframe"):
        if hasattr(report, attr):
            try: return getattr(report, attr)()
            except Exception: pass
    for attr in ("entries", "items", "records", "rows"):
        v = getattr(report, attr, None)
        if v:
            try: return pd.DataFrame(v)
            except Exception: pass
    return pd.DataFrame({"report": [str(report)]})


def _nz(x):
    return None if x is None or (isinstance(x, float) and (np.isnan(x) or np.isinf(x))) else x


def export_dart_rim_excel(model: DartRIMModel, out_dir: Path = EXPORT_DIR, run_date: Optional[date] = None) -> Path:
    run_date = run_date or date.today(); v = model.valuation; tk = model.ticker_dart; name = _corp_name(tk)
    path = out_dir / f"{tk}_{name}_{EXPORT_FILE_TAG}_{run_date:%Y%m%d}.xlsx"
    if path.exists(): path.unlink()

    summary = pd.DataFrame([
        ("종목코드", tk), ("기업명", name), ("데이터 출처", v["data_source"]), ("모형", "RIM (Ohlson 1995) — v5 선형 fade, DART 원천"),
        ("평가일", str(run_date)), ("매출예측 계측일 (forecast_date)", v["forecast_date"]), ("매출예측 모델", v["forecast_model"]),
        ("사용 실적 분기 수", v["revenue_quarters"]),
        ("──── 결과 ────", ""),
        ("현재가 (원)", _nz(v["current_price"])), ("적정주가 (원)", _nz(v["target_price"])), ("Upside (%)", _nz(v["upside_pct"])),
        ("Intrinsic Value (원)", v["intrinsic_value"]), ("  BV₀ (원)", v["bv0"]), ("  BV 출처", v["bv_source"]),
        ("  Σ PV(RI) (원)", v["pv_ri"]), ("  PV(TV) (원)", v["pv_tv"]), ("  TV 비중 (%)", _nz(v["tv_weight_pct"])),
        ("주식수 (추정)", _nz(v["shares"])), ("주식수 추정방식", v["shares_method"]),
        ("──── 자본비용 ────", ""),
        ("Re (CAPM, clip 없음)", v["re"]), ("Rf", RF), ("ERP", ERP), ("β_raw", _nz(v["beta_raw"])), ("β_blume", _nz(v["beta_blume"])),
        ("──── Moat / Phase ────", ""),
        ("Moat 등급 (6-Tier)", v["moat_label"]), ("ω (메타데이터, fade 에 미사용)", v["omega"]),
        ("Phase 1 연수", v["n_phase1"]), ("Phase 2 연수 (선형 fade)", v["n_phase2"]),
        ("EVA spread (ROIC−WACC)", _nz(v["eva_spread"])), ("ROE−Re spread (TTM)", _nz(v["roe_re_spread"])), ("n_pos (20Q)", v["n_positive"]),
        ("ROIC (TTM)", _nz(v["roic"])), ("WACC (EVA 용)", _nz(v["wacc_eva"])), ("과거 TTM ROE", _nz(v["hist_roe_ttm"])),
        ("──── DuPont / 유보 ────", ""),
        ("NPM 계수", model._coefs.get("npm_coef")), ("NPM 방식", model._coefs.get("npm_method")), ("NPM R²", model._coefs.get("npm_r2")),
        ("AT (TTM median)", model._coefs.get("at_median")), ("FL (median)", model._coefs.get("fl_median")),
        ("Retention b₀", v["retention"]), ("b_term (= g_term/Re)", v["b_term"]), ("g_term (fade 기준)", v["g_term_fade"]),
    ], columns=["항목", "값"])

    sched = model.result_df.copy()
    sched.insert(0, "t", range(1, len(sched) + 1))
    dup_rows = []
    wide = model._fs_wide.sort_index()
    for dt in wide.index:
        ni, rv, ta, eq = (wide.loc[dt].get(k, np.nan) for k in ("net_income", "revenue", "total_assets", "total_equity"))
        if all(pd.notna(x) for x in (ni, rv, ta, eq)) and rv > 0 and ta > 0 and eq > 0:
            dup_rows.append({"quarter": f"{dt.year}Q{dt.quarter}", "NI": ni, "Sales": rv, "Assets": ta, "Equity": eq,
                             "NPM": ni / rv, "AT_q": rv / ta, "FL": ta / eq, "ROE_q": ni / eq})
    dupont = pd.DataFrame(dup_rows).tail(40)
    moat = pd.DataFrame([
        ("Exceptional moat", 0.990, 30, ">25%", ">150%", 18, PHASE1_YR_BY_MOAT["Exceptional moat"]),
        ("Wide moat", 0.980, 28, ">15%", ">80%", 16, PHASE1_YR_BY_MOAT["Wide moat"]),
        ("Wide-Narrow moat", 0.970, 25, ">10%", ">40%", 13, PHASE1_YR_BY_MOAT["Wide-Narrow moat"]),
        ("Narrow moat", 0.950, 20, ">6%", ">20%", 10, PHASE1_YR_BY_MOAT["Narrow moat"]),
        ("Some moat", 0.940, 15, ">3%", ">5%", 7, PHASE1_YR_BY_MOAT["Some moat"]),
        ("No moat", 0.880, 10, "else", "else", 0, PHASE1_YR_BY_MOAT["No moat"]),
    ], columns=["grade", "omega", "phase2_yr", "EVA spread 기준", "ROE-Re 기준", "n_pos 최소", "phase1_yr"])
    moat["current"] = moat["grade"].eq(v["moat_label"]).map({True: "◀ 현재", False: ""})

    sales = pd.concat([model._sales_actual.rename("actual"), model._sales_forecast.rename("forecast")], axis=1)
    sales.index = pd.to_datetime(sales.index).date; sales.index.name = "quarter_end"
    try:
        fc_all = pd.read_sql(_sa_text(f"SELECT date, indicator, value FROM {TABLE_DART_FC} WHERE ticker=:t AND forecast_date=:d"),
                             engine, params={"t": tk, "d": v["forecast_date"]})
        if not fc_all.empty:
            pv = fc_all.pivot(index="date", columns="indicator", values="value"); pv.index = pd.to_datetime(pv.index).date
            sales = sales.join(pv, how="outer")
    except Exception:
        pass
    vint = load_forecast_vintages(tk, model.db_info)
    fin = (model._fs_wide / 1e8).round(2); fin.index = pd.to_datetime(fin.index).date; fin.index.name = "quarter_end (억원)"
    assumptions = pd.DataFrame([(k, str(globals()[k])) for k in [
        "FORECAST_QUARTERS", "MIN_DATA_PERIODS", "ENSEMBLE_AGG", "IS_FLOW_MODE", "CF_FLOW_MODE", "MIN_HISTORY", "OLS_MIN_R2", "OLS_MIN_SAMPLES",
        "WINSORIZE_LIMITS", "GDP_GROWTH", "TV_RE_GAP", "RETENTION_FLOOR", "BV_RETENTION_CAP", "PHASE1_BASE_YR", "PHASE1_EXTRA_YR",
        "PHASE1_ROE_FLOOR_RATIO", "PHASE1_ROE_CEILING_RATIO", "ERP_METHOD", "DAMODARAN_ERP_KR", "RD_DEFAULT", "RD_ANNUALIZE_QUARTERLY",
        "DEBT_KEYS", "CASH_KEYS", "FS_UNIT_MULTIPLIER", "FORECAST_MODULE", "TABLE_DART_FS", "TABLE_DART_FC"]], columns=["param", "value"])

    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        summary.to_excel(xw, sheet_name="Summary", index=False)
        sched.to_excel(xw, sheet_name="RI_Schedule", index=False)
        (dupont if not dupont.empty else pd.DataFrame({"info": ["DuPont 데이터 부족"]})).to_excel(xw, sheet_name="DuPont_Q", index=False)
        moat.to_excel(xw, sheet_name="Moat_Tiers", index=False)
        sales.to_excel(xw, sheet_name="Sales")
        (vint if not vint.empty else pd.DataFrame({"info": ["vintage 없음"]})).to_excel(xw, sheet_name="Forecast_Vintages")
        fin.to_excel(xw, sheet_name="Financials_Q")
        (model._mapping_df if model._mapping_df is not None else pd.DataFrame()).to_excel(xw, sheet_name="Account_Mapping", index=False)
        _quality_df(model.report).to_excel(xw, sheet_name="Quality", index=False)
        assumptions.to_excel(xw, sheet_name="Assumptions", index=False)
        for ws in xw.book.worksheets:
            ws.freeze_panes = "B2"
            for col in ws.columns:
                w = max((len(str(c.value)) for c in col if c.value is not None), default=8)
                ws.column_dimensions[col[0].column_letter].width = min(max(10, w + 2), 60)
    return path

## 실행
### Step A · (선택) 개별 종목 점검 — DART 계정 매핑 + 예측 미리보기 (DB 저장 X)

In [9]:
INSPECT_TICKER = TICKERS[0]
_w, _mp = show_dart_account_coverage(INSPECT_TICKER, db_info)
_ins = inspect_dart_ticker(INSPECT_TICKER, db_info)

[16:03:02][FLOW] IS(revenue): median (Q1+H1+Q3)/FY = 0.68 (n=6) → quarter
[16:03:02][FLOW] CF(da_cf): median (Q1+H1+Q3)/FY = 1.40 (n=2) → cumulative
[16:03:03][278470] DART wide shape=(27, 28)  2019-12-31~2026-06-30  표준키 28/37 매핑  IS=quarter CF=cumulative

[표준키 매핑]
             std_key                                                                    component match    flow       mode  n_q
             revenue                                                            ifrs-full_Revenue    id      is    quarter   26
       cost_of_sales                                                        ifrs-full_CostOfSales    id      is    quarter   26
    operating_income                                                     dart_OperatingIncomeLoss    id      is    quarter   24
       pretax_income                                                ifrs-full_ProfitLossBeforeTax    id      is    quarter   26
         tax_expense                               ifrs-full_IncomeTaxExpenseContinuingOperati

### Step B · 매출 예측 → `korea_revenue_forecast_from_DART` (FCFF 노트북과 공유)

같은 계측일에 FCFF 노트북에서 이미 예측했다면 `RERUN_FORECAST=False` 로 두면 재사용합니다.

In [10]:
fc_result = run_dart_forecast_batch(TICKERS, db_info, forecast_date=FORECAST_DATE, rerun=RERUN_FORECAST, verbose=VERBOSE)

DART 매출예측:   0%|          | 0/1 [00:00<?, ?it/s]

[16:03:13][FLOW] IS(revenue): median (Q1+H1+Q3)/FY = 0.68 (n=6) → quarter
[16:03:13][FLOW] CF(da_cf): median (Q1+H1+Q3)/FY = 1.40 (n=2) → cumulative
[16:03:13][278470] DART wide shape=(27, 28)  2019-12-31~2026-06-30  표준키 28/37 매핑  IS=quarter CF=cumulative
[메모리] forecast_sarima 실행 전: 484.98 MB
[메모리] find_best_sarima_params 실행 전: 484.98 MB
[메모리] find_best_sarima_params 실행 후: 485.02 MB (변화: +0.04 MB)
[메모리] forecast_sarima 실행 후: 485.02 MB (변화: +0.04 MB)
[메모리] forecast_ets 실행 전: 485.02 MB
[메모리] forecast_ets 실행 후: 485.02 MB (변화: +0.00 MB)
[메모리] forecast_theta 실행 전: 485.02 MB
[메모리] forecast_theta 실행 후: 485.02 MB (변화: +0.00 MB)
  저장 완료 | 32행 → korea_revenue_forecast_from_DART  (forecast_date=2026-08-25)
[예측 완료] 성공 1 / 실패 0  경과 10s


### Step C · RIM Valuation → Excel (DB 저장 없음)

In [11]:
run_date = date.today()
_summary_rows, t0 = [], time.time()
print(f"\n{'='*70}\n[DART RIM Valuation] {len(TICKERS)}종목  run_date={run_date}  → {EXPORT_DIR}\n{'='*70}")
for i, tk in enumerate(TICKERS, 1):
    print(f"\n[{i}/{len(TICKERS)}] {tk} " + "─" * 50)
    try:
        _m = DartRIMModel(ticker=tk, engine=engine, db_info=db_info, rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX,
                          forecast_date=None, verbose=VERBOSE).run()
        _p = export_dart_rim_excel(_m, EXPORT_DIR, run_date)
        v = _m.valuation
        tp = f"{v['target_price']:,.0f}원" if not np.isnan(v["target_price"]) else "N/A"
        cp = f"{v['current_price']:,.0f}원" if not np.isnan(v["current_price"]) else "N/A"
        up = f"{v['upside_pct']:+.1f}%" if not np.isnan(v["upside_pct"]) else "N/A"
        print(f"  ✓ Saved: {_p}")
        print(f"    적정주가 {tp}   현재가 {cp}   Upside {up}   Re {v['re']:.2%}")
        print(f"    BV₀ {v['bv0']/1e12:.2f}조 [{v['bv_source']}]  ΣPV(RI) {v['pv_ri']/1e12:.2f}조  PV(TV) {v['pv_tv']/1e12:.2f}조  IV {v['intrinsic_value']/1e12:.2f}조")
        print(f"    Moat {v['moat_label']} (Ph1={v['n_phase1']}yr, Ph2={v['n_phase2']}yr, b₀={v['retention']:.2f})   주식수: {v['shares_method']}   forecast_date={v['forecast_date']}")
        _summary_rows.append({"ticker": tk, "status": "ok", "excel": str(_p), "target_price": v["target_price"], "current_price": v["current_price"],
                              "upside_pct": v["upside_pct"], "re": v["re"], "moat": v["moat_label"], "bv_source": v["bv_source"],
                              "forecast_date": v["forecast_date"]})
    except Exception as e:
        print(f"  ✗ FAIL: {e}"); traceback.print_exc()
        _summary_rows.append({"ticker": tk, "status": "fail", "excel": "", "target_price": np.nan, "current_price": np.nan,
                              "upside_pct": np.nan, "re": np.nan, "moat": "?", "bv_source": "", "forecast_date": None, "msg": str(e)[:150]})
    finally:
        clear_memory()
print(f"\n{'='*70}\n[완료] OK={sum(r['status']=='ok' for r in _summary_rows)} FAIL={sum(r['status']=='fail' for r in _summary_rows)}  경과 {time.time()-t0:.0f}s\n{'='*70}")
summary_df = pd.DataFrame(_summary_rows).set_index("ticker")
display(summary_df)


[DART RIM Valuation] 1종목  run_date=2026-08-25  → C:\Users\82108\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터

[1/1] 278470 ──────────────────────────────────────────────────
[16:03:23][FLOW] IS(revenue): median (Q1+H1+Q3)/FY = 0.68 (n=6) → quarter
[16:03:23][FLOW] CF(da_cf): median (Q1+H1+Q3)/FY = 1.40 (n=2) → cumulative
[16:03:23][278470] DART wide shape=(27, 28)  2019-12-31~2026-06-30  표준키 28/37 매핑  IS=quarter CF=cumulative
[16:03:23][278470] Sales actual=26Q forecast=8Q (Ensemble, 2026-08-25)
[16:03:23][FLOW] IS(revenue): median (Q1+H1+Q3)/FY = 0.68 (n=6) → quarter
[16:03:23][FLOW] CF(da_cf): median (Q1+H1+Q3)/FY = 1.40 (n=2) → cumulative
[16:03:24][278470] DART wide shape=(27, 28)  2019-12-31~2026-06-30  표준키 28/37 매핑  IS=quarter CF=cumulative
[16:03:26][278470] Re=11.335% β_raw=nan β_blume=1.000
[16:03:26][278470] DuPont NPM=0.2057(ols) AT=1.983 FL=1.77
[16:03:30][278470] EVA: ROIC=73.65% WACC=11.27% spread=+62.38% n_pos=19/20 ROE-Re=+77.4%
[16:03:30][278470] Retention=0.731 payout=0.2

,status,excel,target_price,current_price,upside_pct,re,moat,bv_source,forecast_date
ticker,,,,,,,,,
278470,ok,C:\Users\82108\OneDrive\INVESTMENT\한국주식\FCFF_R...,2.024930e+07,333500.0,5971.754553,0.11335,Exceptional moat,Equity,2026-08-25


### Step D · 계측일별 매출 예측 변화 추적

In [12]:
VINTAGE_TICKER = TICKERS[0]
_v = load_forecast_vintages(VINTAGE_TICKER, db_info, indicator="Ensemble")
if _v.empty:
    print(f"[{VINTAGE_TICKER}] 저장된 vintage 없음")
else:
    print(f"[{VINTAGE_TICKER}] Ensemble 예측 (억원) — 계측일 × 대상분기")
    _disp = _v.copy(); num = _disp.columns.drop("last_actual"); _disp[num] = (_disp[num] / 1e8).round(0)
    display(_disp)

[278470] Ensemble 예측 (억원) — 계측일 × 대상분기


date,last_actual,2026Q3,2026Q4,2027Q1,2027Q2,2027Q3,2027Q4,2028Q1,2028Q2
forecast_date,,,,,,,,,
2026-08-25,2026-06-30,8921.0,13051.0,14738.0,18342.0,22124.0,31729.0,35831.0,44592.0


In [13]:
df = load_dart_long("278470", db_info)
df[(df.bsns_year == 2026) & (df.sj_div.isin(["IS","CIS"]))][["quarter","sj_div","account_id","account_nm","thstrm_amount"]]

,quarter,sj_div,account_id,account_nm,thstrm_amount
3081,H1,CIS,dart_OtherOperatingExpense,기타영업비용,6.877728e+09
3082,H1,CIS,dart_OtherOperatingIncome,기타영업수익,1.196932e+10
3083,H1,CIS,dart_TotalSellingGeneralAdministrativeExpenses,판매비와 관리비,4.223245e+11
3085,H1,CIS,ifrs-full_ComprehensiveIncome,총포괄손익,1.414535e+11
3086,H1,CIS,ifrs-full_ComprehensiveIncomeAttributableToNon...,비지배지분,0.000000e+00
3087,H1,CIS,ifrs-full_ComprehensiveIncomeAttributableToOwn...,지배기업 소유주지분,1.414535e+11
3088,H1,CIS,ifrs-full_CostOfSales,매출원가,1.597473e+11
3089,H1,CIS,ifrs-full_BasicEarningsLossPerShare,보통주 기본주당이익,3.711000e+03
3090,H1,CIS,ifrs-full_DilutedEarningsLossPerShare,보통주 희석주당이익,3.711000e+03
3091,H1,CIS,ifrs-full_GrossProfit,매출총이익,6.077860e+11


In [15]:
sql = _sa_text(f"""
    SELECT id, bsns_year, quarter, sj_div, account_id, account_nm, thstrm_amount
    FROM {TABLE_DART_FS}
    WHERE ticker = '278470' AND bsns_year = 2026 AND sj_div IN ('IS','CIS')
    ORDER BY id
""")
with engine.connect() as c:
    chk = pd.read_sql(sql, c)

chk.to_csv(EXPORT_DIR / "278470_2026_IS_rows.csv", index=False, encoding="utf-8-sig")
print(chk.to_string(index=False))

     id  bsns_year quarter sj_div                                                                                  account_id                account_nm  thstrm_amount
8634208       2026      Q1    CIS                                                                  dart_OtherOperatingExpense                    기타영업비용   4.317564e+09
8634209       2026      Q1    CIS                                                                   dart_OtherOperatingIncome                    기타영업수익   1.102179e+10
8634210       2026      Q1    CIS                                              dart_TotalSellingGeneralAdministrativeExpenses                  판매비와 관리비   3.113455e+11
8634212       2026      Q1    CIS                                                               ifrs-full_ComprehensiveIncome                     총포괄손익   1.172751e+11
8634213       2026      Q1    CIS                          ifrs-full_ComprehensiveIncomeAttributableToNoncontrollingInterests                     비지배지분   0.000000e+0